In [1]:
# 1. LIBRERÍAS Y CONFIGURACIÓN
import pandas as pd
from pathlib import Path

# 2. RUTAS DE LOS ARCHIVOS
# Ruta del archivo objetivo al que le queremos pegar la info
ruta_periodo = Path(r"D:\ProyectoAnalisisElectrico\DiaPromedio\Periodos\2505_2604\2505_2604_mean_period.parquet")

# Ruta de nuestra "fuente de la verdad" geográfica
ruta_infra = Path(r"D:\ProyectoAnalisisElectrico\BarrasEstaciones\infraestructura_consolidada.parquet")

print("Rutas configuradas correctamente.")

Rutas configuradas correctamente.


In [2]:
# Cargar los datos
print("Cargando archivos...")
df_periodo = pd.read_parquet(ruta_periodo)
df_infra = pd.read_parquet(ruta_infra)

print(f"Filas en el archivo de periodos: {len(df_periodo)}\n")
print(df_periodo.columns)
# Seleccionar solo las columnas que necesitamos pegar
columnas_geo = ['nombre_barra', 'tension', 'region', 'macrozona', 'confianza', 'IA']
df_infra_sub = df_infra[columnas_geo]

# 1. Contar registros ANTES de limpiar
total_antes = len(df_infra_sub)

# 2. Eliminar duplicados (usando nombre_barra y tension como llave)
df_infra_geo = df_infra_sub.drop_duplicates(subset=['nombre_barra', 'tension'])

# 3. Contar registros DESPUÉS y calcular cuántos se borraron
total_despues = len(df_infra_geo)
duplicados_eliminados = total_antes - total_despues

print("--- REVISIÓN DE DUPLICADOS EN BASE MAESTRA ---")
print(f"Total de registros iniciales en infraestructura: {total_antes}")
print(f"Duplicados eliminados: {duplicados_eliminados}")
print(f"Total de registros únicos listos para el cruce: {total_despues}\n")

Cargando archivos...
Filas en el archivo de periodos: 163056

Index(['clave', 'Zona', 'Hora', 'medida', 'CMg[CLP/KWh]', 'valorizado_CLP',
       'Calendario_Activo', 'RUT', 'rut_log', 'n_ruts', 'Razon_Social',
       'razon_social_log', 'n_razones_sociales', 'Nombre_Corto',
       'nombre_corto_log', 'n_nombres_cortos', 'nombre_barra',
       'nombre_barra_log', 'n_nombres_barra', 'tension', 'tension_log',
       'n_tensiones', 'tipo', 'period', 'medida_total'],
      dtype='object')
--- REVISIÓN DE DUPLICADOS EN BASE MAESTRA ---
Total de registros iniciales en infraestructura: 1420
Duplicados eliminados: 0
Total de registros únicos listos para el cruce: 1420



In [3]:
# Realizar el cruce (Left Join asegura que no perdamos datos de df_periodo)
df_enriquecido = pd.merge(
    df_periodo,
    df_infra_geo,
    on=['nombre_barra', 'tension'], # Usamos ambas como llave para evitar ambigüedades
    how='left'
)

# Revisión rápida para ver si quedaron barras sin cruzar (huérfanas totales)
nulos_post_cruce = df_enriquecido['macrozona'].isna().sum()

print("\n--- RESULTADO DEL CRUCE ---")
print(f"Filas originales: {len(df_periodo)}")
print(f"Filas después del cruce: {len(df_enriquecido)}")
print(f"Barras que no encontraron coincidencia geográfica: {nulos_post_cruce}")

# Visualizar el resultado final
print(df_enriquecido.columns)
df_enriquecido.head()



--- RESULTADO DEL CRUCE ---
Filas originales: 163056
Filas después del cruce: 163056
Barras que no encontraron coincidencia geográfica: 0
Index(['clave', 'Zona', 'Hora', 'medida', 'CMg[CLP/KWh]', 'valorizado_CLP',
       'Calendario_Activo', 'RUT', 'rut_log', 'n_ruts', 'Razon_Social',
       'razon_social_log', 'n_razones_sociales', 'Nombre_Corto',
       'nombre_corto_log', 'n_nombres_cortos', 'nombre_barra',
       'nombre_barra_log', 'n_nombres_barra', 'tension', 'tension_log',
       'n_tensiones', 'tipo', 'period', 'medida_total', 'region', 'macrozona',
       'confianza', 'IA'],
      dtype='object')


,clave,Zona,Hora,medida,CMg[CLP/KWh],valorizado_CLP,Calendario_Activo,RUT,rut_log,n_ruts,...,tension,tension_log,n_tensiones,tipo,period,medida_total,region,macrozona,confianza,IA
0,$C$439,Norte,0,-16265.098843,66.273480,-1.068584e+06,111111111111,96.505.760-9,96.505.760-9,1,...,220,220,1,L,2505_2604,-374611.038665,Antofagasta,Norte Grande,100.0,False
1,$C$439,Norte,1,-16018.128504,65.684023,-1.044962e+06,111111111111,96.505.760-9,96.505.760-9,1,...,220,220,1,L,2505_2604,-374611.038665,Antofagasta,Norte Grande,100.0,False
2,$C$439,Norte,2,-15406.094948,66.415769,-1.018780e+06,111111111111,96.505.760-9,96.505.760-9,1,...,220,220,1,L,2505_2604,-374611.038665,Antofagasta,Norte Grande,100.0,False
3,$C$439,Norte,3,-15885.848474,66.879940,-1.053275e+06,111111111111,96.505.760-9,96.505.760-9,1,...,220,220,1,L,2505_2604,-374611.038665,Antofagasta,Norte Grande,100.0,False
4,$C$439,Norte,4,-15792.851028,66.719561,-1.045057e+06,111111111111,96.505.760-9,96.505.760-9,1,...,220,220,1,L,2505_2604,-374611.038665,Antofagasta,Norte Grande,100.0,False


In [4]:
ruta_salida = ruta_periodo.parent / "2505_2604_mean_period_loc.parquet"

df_enriquecido.to_parquet(ruta_salida, engine="pyarrow", compression="snappy")

print(f"Archivo enriquecido guardado exitosamente en:\n{ruta_salida}")

Archivo enriquecido guardado exitosamente en:
D:\ProyectoAnalisisElectrico\DiaPromedio\Periodos\2505_2604\2505_2604_mean_period_loc.parquet
